# Second-to-last-turn policy comparison

Compare a second-to-last throw selected by the value network with one selected by immediate actual-score grid search. Both policies use actual-score grid search for the last throw and start from the same saved sheet states.

In [3]:
import importlib
import logging
from pathlib import Path
import sys

logging.basicConfig(level=logging.ERROR, format='%(message)s')
logging.getLogger('matplotlib').setLevel(logging.WARNING)

repo_root = Path.cwd()
if not (repo_root / 'curling_nn.py').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import bot
import curling_nn
import evaluation
import polars as pl
value_network_weights_path = repo_root / 'weights/value_network_weights.npz'
importlib.reload(evaluation)

<module 'evaluation' from '/home/vietafan/curling/evaluation.py'>

In [4]:
states = evaluation.load_sheet_states(
    repo_root / 'scratch/second_to_last_evaluation_states.npz'
)
value_network, value_normalizer = curling_nn.load_v_weights(
    value_network_weights_path
)
if value_network.num_stones != 9:
    raise ValueError(
        'This comparison requires the new 9-stone value network; '
        f'loaded {value_network.num_stones}-stone weights.'
    )

comparison = evaluation.compare_second_to_last_policies(
    states,
    second_to_last_team=0,
    throw_searcher=bot.ThrowsGridSearcher(10, 10, 4),
    value_network=value_network,
    value_normalizer=value_normalizer,
)
comparison

sim_idx,second_to_last_policy,team_0_score,team_1_score,team_0_net_score
i64,str,i64,i64,i64
0,"""value_network""",0,1,-1
1,"""value_network""",0,1,-1
2,"""value_network""",0,2,-2
3,"""value_network""",0,2,-2
4,"""value_network""",0,1,-1
…,…,…,…,…
295,"""grid_search""",0,1,-1
296,"""grid_search""",0,1,-1
297,"""grid_search""",0,1,-1


In [8]:
summary = (
    comparison.group_by('second_to_last_policy')
    .agg([
        pl.col('team_0_score').mean().alias('mean_team_0_score'),
        pl.col('team_1_score').mean().alias('mean_team_1_score'),
        pl.col('team_0_net_score').mean().alias('mean_team_0_net_score'),
        (pl.col('team_0_net_score').std() / pl.col('team_0_net_score').count() ** .5).alias('mean_team_0_net_score_stderr')
    ])
)
summary

second_to_last_policy,mean_team_0_score,mean_team_1_score,mean_team_0_net_score,mean_team_0_net_score_stderr
str,f64,f64,f64,f64
"""value_network""",0.006667,1.65,-1.643333,0.047082
"""grid_search""",0.01,1.806667,-1.796667,0.073349


In [18]:
comparison.pivot(
    on="second_to_last_policy", index="sim_idx", values="team_0_net_score"
).with_columns(score_diff=pl.col("value_network") - pl.col("grid_search")).group_by(
    "score_diff"
).agg(
    pl.col("sim_idx").count()
).sort(
    by="score_diff"
)

score_diff,sim_idx
i64,u32
-3,2
-2,11
-1,59
0,148
1,38
2,33
3,7
4,2


In [20]:
comparison.pivot(
    on="second_to_last_policy", index="sim_idx", values="team_0_net_score"
).with_columns(score_diff=pl.col("value_network") - pl.col("grid_search")).select((pl.col("score_diff").sum() / (pl.col("score_diff")**2).sum() ** .5))

score_diff
f64
2.341338
